# Python. Работа с JSON и XML

## Мотивация

Данные почти никогда не лежат готовым массивом чисел. Обучающая выборка
приходит JSON-ответом API, разметка от подрядчика — XML-выгрузкой, а половина
нужных цифр вообще осталась только на веб-странице. Прежде чем что-то
посчитать, эти данные надо **достать, проверить и сохранить** — и сделать так,
чтобы завтра тот же скрипт отработал повторно и дал тот же результат.

Сегодня мы пройдём этот путь целиком: прочитаем JSON и XML, отличим сломанный
синтаксис от неожиданной структуры, сходим за данными по HTTP, разберём четыре
разных смысла слова «ошибка», выдернем текст из кривого HTML и в конце
поговорим с языковой моделью через тот же самый REST API. Сквозные примеры —
каталог книг (он же входные данные задач B5 и M1) и список запусков
эксперимента с неполными метриками.

Что вы унесёте с занятия: привычку **не доверять входным данным** и знание,
какой инструмент брать под какой формат.

## 0. Подготовка

`json`, `xml.etree.ElementTree` и `os` входят в стандартную библиотеку
Python — ставить ничего не надо. Внешние пакеты нужны для HTTP, HTML,
OpenAI-совместимого API и учебного сервера из задач:

```bash
mkdir -p ~/seminar-11                       # рабочий каталог курса, если его ещё нет
cp -r <репозиторий>/seminars/11-python-json-xml/. ~/seminar-11/
cd ~/seminar-11
uv init --python 3.12    # создаёт pyproject.toml; если проект уже есть — пропустите
uv add jupyter requests beautifulsoup4 openai fastapi uvicorn
```

Имя пакета и имя импорта иногда различаются: пакет `beautifulsoup4`
импортируется как `bs4`. Ноутбук запускайте из этого же окружения
(`uv run jupyter lab`), иначе импорты не найдутся — как в семинаре 5.

Всё, что создаёт демка, кладём в `~/seminar-11/demo`. Не в `/tmp`: он
вычищается при перезагрузке, а файлы пригодятся и после занятия — на них
удобно проверять свои решения задач.

In [ ]:
import os

# Рабочий каталог демонстрации: он же останется у вас после занятия.
WORK = os.path.expanduser("~/seminar-11/demo")
os.makedirs(WORK, exist_ok=True)   # exist_ok — повторный запуск ячейки не упадёт
print(WORK)                        # печатаем, чтобы знать, где потом искать файлы

Две служебные функции: короткое имя файла в рабочем каталоге и показ
текстового файла целиком. Это не про тему семинара — просто чтобы дальше
примеры не заслонялись возёй с путями.

In [ ]:
def work_path(name):
    """Путь к файлу демонстрации: work_path("students.json")."""
    return os.path.join(WORK, name)

def show_text(path):
    """Показать небольшой текстовый файл целиком — как он лежит на диске."""
    with open(path, encoding="utf-8") as file:
        print(file.read())

## 1. JSON: зачем он нужен и как его читать

Обычный TXT хранит только символы. Запишем `Анна 86 true` — и программа уже не
знает, где имя, где балл, текст ли `true` и как дописать второго студента.
Свои разделители придумать можно, но тогда у каждого автора получится
собственный формат, а читать их будет нечем.

JSON — тоже текст, его так же можно открыть в редакторе и отправить по сети,
но структура и типы записываются по общим правилам. У полей есть имена,
объекты и списки вкладываются друг в друга, а число, строка, логическое
значение и «значения нет» — разные вещи. Поэтому один файл одинаково понимают
Python, JavaScript и почти всё остальное. Цена — строгий синтаксис и несколько
лишних символов.

Соответствие JSON и Python:

| JSON | Python |
|---|---|
| объект `{...}` | словарь `dict` |
| массив `[...]` | список `list` |
| строка, число | `str`, `int` или `float` |
| `true`, `false` | `True`, `False` |
| `null` | `None` |

Четыре функции, и различаются они буквой `s` — от **string**:

- `json.loads(text)` — разобрать JSON-строку;
- `json.dumps(data)` — получить JSON-строку;
- `json.load(file)` — прочитать JSON из уже *открытого* файла;
- `json.dump(data, file)` — записать JSON в уже *открытый* файл.

<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

JSON придумал Дуглас Крокфорд около 2001 года — не изобрёл, а
*выделил*: это буквально подмножество синтаксиса литералов JavaScript, которое
он объявил форматом обмена. Никакого комитета и стандарта поначалу не было,
был сайт json.org с одной картинкой синтаксиса — и именно за счёт того, что
формат помещался на одну страницу, он и вытеснил XML из веб-API.

Крокфорд же в лицензию своей эталонной реализации вписал строчку «The Software
shall be used for Good, not Evil». Юристы крупных компаний потом отдельно
просили разрешения использовать библиотеку во зло — IBM такое разрешение
формально получила.

</details>

Начинаем с обычного словаря Python. Никакого «JSON-объекта» в языке нет:
есть словари, списки и простые типы — их-то и сериализуют.

In [ ]:
# Обычная структура Python: словарь, внутри — список словарей.
students = {
    "course": "Linux и Python",
    "active": True,                       # станет true — в JSON у логического свой тип
    "students": [
        {"name": "Анна", "score": 86},
        {"name": "Илья", "score": 73},
    ],
}

Теперь записываем словарь в файл. Два необязательных аргумента, которые
меняют только вид файла для человека — без них JSON тоже полностью валиден и
программой читается одинаково:

- `ensure_ascii=False` — оставить кириллицу буквами; по умолчанию `json`
  экранирует её в `\u0410\u043d\u043d\u0430`, и файл становится нечитаемым
  глазами;
- `indent=2` — расставить отступы (годится любое число); без него всё уедет в
  одну строку.

На разбор это не влияет никак, но diff в git и чтение чужого файла — влияет
сильно, поэтому в курсе пишем с ними.

In [ ]:
import json

path = work_path("students.json")
with open(path, "w", encoding="utf-8") as file:
    json.dump(students, file, ensure_ascii=False, indent=2)

show_text(path)      # смотрим глазами, что реально легло на диск

Файл на диске — это текст. Читаем его обратно и убеждаемся, что получили
привычные объекты Python, а не какой-то особый тип.

In [ ]:
with open(path, encoding="utf-8") as file:
    loaded = json.load(file)

print(type(loaded), type(loaded["students"]))   # dict и list — обычные типы Python
print(loaded["students"][0]["score"])           # 86 — уже число, а не строка "86"
print(loaded["active"] is True)                 # true из файла стал настоящим True
print(json.dumps(loaded["students"][0], ensure_ascii=False))  # dumps — обратно в строку
print(json.dumps(loaded["students"][0]))        # без ensure_ascii=False: те самые \uXXXX

json.load, loads, dump и dumps: файл против строки

Четыре имени различаются ровно двумя вещами: читаем мы или пишем, и работаем с
открытым файлом или со строкой. Перепутанная пара падает сразу и говорит об
этом прямым текстом: `AttributeError: 'str' object has no attribute 'read'`
означает, что в `load` отдали строку вместо файла.

#### ❓ **Вопрос**: Что возвращают `json.load(file)` и `json.loads(text)` и почему работает цепочка `loaded["students"][0]["score"]`?

<details>

<summary><strong>Ответ</strong></summary>

Обе функции возвращают обычные объекты Python и различаются только источником: `load` читает из открытого файла, `loads` — из строки. Верхний уровень нашего файла — объект, он стал `dict`; по ключу `students` лежит массив, он стал `list`; его элементы — снова объекты, то есть `dict`. Поэтому к ним применимы обычные ключи и индексы Python, а `score` приходит числом — в JSON у чисел свой тип.

</details>

## 2. Две разные беды: сломанный синтаксис и неожиданная структура

Слово «битый JSON» скрывает две совершенно разные ситуации, и лечатся они
по-разному.

1. **Синтаксис.** Не закрыта скобка, одинарные кавычки вместо двойных, лишняя
   запятая, два значения верхнего уровня подряд. Парсер до данных вообще не
   доберётся и бросит `json.JSONDecodeError` с номером строки и столбца.
   Важно: парсер останавливается на **первой** же встреченной проблеме и
   списка всех ошибок не собирает — починили одну, запускайте снова, чтобы
   увидеть следующую.
2. **Структура.** Файл разобрался, но вместо списка пришёл объект, пропало
   поле `loss` или число приехало строкой. Парсеру всё равно — проверять
   должна ваша программа.

две точки отказа: разбор и работа с полями

Начнём с первой. Обратите внимание на сообщение: парсер называет место.

In [ ]:
broken = '{"name": "Анна", "score": 86,}'   # лишняя запятая после последнего поля

try:
    json.loads(broken)
except json.JSONDecodeError as error:
    print(error)          # в сообщении есть и причина, и позиция (line/column)

<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

«Лишняя запятая» (trailing comma) — самая частая правка в чужих
конфигах, и запрещена она не случайно: Крокфорд сознательно оставил в JSON
только то, что одинаково разбирают все языки. Зато вокруг развелись
диалекты — JSON5, JSONC (это он в `settings.json` VS Code), HJSON, — где
разрешены и запятые, и комментарии. Мораль для студента: если файл называется
`.json`, но парсер на него ругается, посмотрите, не «почти-JSON» ли это; и
никогда не дописывайте комментарии в файл, который читает `json.load`.

</details>

Со второй бедой сложнее: тут никто не ругнётся. Возьмём четыре записи об
экспериментах — синтаксически безупречные, но неполные. У `beta` нет
`accuracy`, у `gamma` нет `loss`, а у `delta` вместо объекта `metrics` стоит
`null`.

In [ ]:
text = """[
  {"name": "alpha", "metrics": {"loss": 0.31, "accuracy": 0.91}},
  {"name": "beta",  "metrics": {"loss": 0.27}},
  {"name": "gamma", "metrics": {"accuracy": 0.88}},
  {"name": "delta", "metrics": null}
]"""

runs = json.loads(text)      # разбирается без единой жалобы
print(len(runs), runs[3])    # 4 записи; у delta metrics — None, ключ-то на месте

Наивный вариант `runs[2]["metrics"]["loss"]` уронил бы обработку на
третьей записи — и первые две тоже пропали бы. Так делать нельзя: одна битая
строка в выгрузке не должна стоить всего файла.

Правило простое: **необязательное** поле читаем через `.get()` — он вернёт
`None` вместо исключения, — а **обязательное** оставляем в квадратных скобках,
чтобы его пропажа не осталась незамеченной. Здесь `metrics` и `loss`
необязательны, а `name` мы считаем обязательным и берём как `run["name"]`.

Одного `.get()` мало. Заманчивая короткая запись `run.get("metrics", {}).get("loss")`
на `delta` упадёт с `AttributeError`: значение по умолчанию `{}` подставляется,
только когда **ключа нет**, а у нас ключ есть и в нём лежит `None`. Поэтому
перед вторым `.get()` проверяем тип — `isinstance`. Эта же ловушка ждёт вас в
задаче B3, там `"values": null`.

In [ ]:
valid_runs, errors = [], []
for index, run in enumerate(runs):
    metrics = run.get("metrics")                 # может не быть ключа, а может быть None
    loss = metrics.get("loss") if isinstance(metrics, dict) else None
    if type(loss) in (int, float):               # именно type, а не isinstance — см. ниже
        valid_runs.append({"name": run["name"], "loss": loss})
    else:
        errors.append({"index": index, "reason": "нет числового loss"})

Смотрим результат: обработка дошла до конца, а брак не потерялся — он
сложен отдельно и с номером записи, по которому его можно найти в исходном
файле.

In [ ]:
print("годные:", valid_runs)
print("брак:  ", errors)

<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

Почему `type(loss) in (int, float)`, а не привычное
`isinstance(loss, (int, float))`. В Python `bool` — подкласс `int`, поэтому
`isinstance(True, int)` истинно, и запись `{"loss": true}` тихо прошла бы
валидацию как «число 1». На реальной выгрузке это выглядит так: метрика
внезапно равна 1.0 у сотни экспериментов, а это на самом деле были флаги.

Исторически так вышло потому, что до Python 2.3 логического типа вообще не
было — писали `1` и `0`, — и когда `bool` добавили, его сделали подклассом
`int`, чтобы не сломать весь существующий код. В обычном коде пишите
`isinstance`, но в валидации чужих данных разница вылезает.

</details>

#### ❓ **Вопрос**: В файле пришло `{"name": "delta", "metrics": {"loss": true}}`. Попадёт ли эта запись в `valid_runs`, и что было бы с `isinstance`?

<details>

<summary><strong>Ответ</strong></summary>

С текущей проверкой не попадёт: `type(True)` — это `bool`, а `bool` в кортеже `(int, float)` не перечислен. Запись уедет в `errors`. Если заменить проверку на `isinstance(loss, (int, float))`, она пройдёт как годная и `loss` станет равен `True`, потому что `bool` — подкласс `int`; дальше среднее значение метрики посчитается по флагам.

</details>

#### ❓ **Вопрос**: Что произойдёт при `json.loads(text)`, если `text` выглядит так, и сообщит ли парсер обо всех проблемах сразу?

```python
text = """
[
  {"name": "alpha"},
], [ {"name": "beta"} ]
"""
```

<details>

<summary><strong>Ответ</strong></summary>

Будет `JSONDecodeError`, и только про первую проблему: парсер останавливается на первом же нарушении, а не собирает список. Сначала он споткнётся на запятой перед `]`. Уберём её — вылезет вторая ошибка: в документе два значения верхнего уровня, а JSON допускает ровно одно. Чинится либо одним общим массивом, либо массивом из двух вложенных.

</details>

## 3. XML: то же самое, но деревом

JSON и XML: одна и та же книга в двух моделях данных

XML — текстовое дерево элементов. В `<book id="b1"><title>Python</title></book>`
`book` и `title` — элементы (теги), `id` — атрибут, `Python` — текст, а
`title` — дочерний элемент `book`.

| | JSON | XML |
|---|---|---|
| Модель | объекты, массивы, простые типы | дерево элементов, атрибутов и текста |
| Числа и `true` | имеют собственный тип | без схемы всё читается как строка |
| Объём | обычно компактнее | обычно многословнее |
| Сильная сторона | API и данные приложений | документы, порядок, namespaces, отраслевые схемы |
| Где встретите | REST API, конфиги, обмен между сервисами | SOAP, docx/xlsx, SVG, Maven, госсистемы |

Ни один формат не «новая версия» другого. JSON удобнее для обычных структур
приложения; XML берут, когда важны порядок и разметка документа, атрибуты,
пространства имён или когда отраслевой стандарт уже написан в XML и выбора
нет.

<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

В нулевые XML был ответом на всё: конфиги, протоколы (SOAP), сборка
(Ant, Maven), даже вёрстка. Отсюда шутка тех лет: «XML — как насилие: если не
помогло, значит, мало применили». Развернуться назад заставил веб — браузеру
надо было разбирать ответ на JavaScript, а `eval` над JSON стоил одну строчку
против целого DOM-парсера.

Но хоронить XML рано, и вы в этом убедитесь на первой же работе с реальными
данными: любой `.docx` и `.xlsx` — это zip-архив с XML внутри, SVG-картинки в
`images/` этого семинара тоже XML, и вся государственная отчётность в России
до сих пор сдаётся XML-выгрузками по XSD-схемам.

</details>

Возьмём каталог книг — он же будет входными данными в задачах B5 и M1.
Обратите внимание: у `b1` есть атрибут `lang`, у `b2` его нет.

In [ ]:
xml_text = """<library>
  <catalog kind="current">
    <book id="b1" lang="ru"><title>Python</title><price>1200</price></book>
    <book id="b2"><title>Linux</title><price>900</price></book>
  </catalog>
  <catalog kind="archive"><book id="b3"><title>Old Unix</title><price>1500</price></book></catalog>
</library>"""

Разбираем строку в дерево. `ET.fromstring(text)` читает из строки,
`ET.parse(path)` — из файла (у него потом нужен `.getroot()`).

In [ ]:
import xml.etree.ElementTree as ET

root = ET.fromstring(xml_text)                    # корневой элемент library
print(root.tag)                                   # имя тега корня
print([child.get("kind") for child in root])      # по элементу итерируемся как по списку детей

Теперь выбираем нужное. `ElementTree` понимает полезное подмножество
XPath — языка адресации внутри XML-дерева:

- `find(path)` — первый совпавший элемент или `None`;
- `findall(path)` — список всех совпавших;
- `findtext(path)` — текст первого совпавшего; `None`, если ничего не нашлось,
  поэтому оборачивать его в `int()`/`float()` вслепую нельзя;
- `.//book` — искать на любой глубине, `./catalog/book` — по конкретному пути;
- `[@kind='current']` — фильтр по атрибуту;
- `element.get("id")` — прочитать атрибут, второй аргумент — значение по умолчанию.

Это **подмножество** XPath, и границу видно быстро: сравнений вроде
`book[price > 1000]`, функций (`contains`, `starts-with`) и выборки текста
`.../title/text()` в `ElementTree` нет. Именно поэтому задача M1 просит
поставить `lxml` — в нём XPath полный. Пример — в «Дополнительно», раздел
«Полный XPath через lxml».

In [ ]:
# Только книги из текущего каталога: путь начинается от корня.
for book in root.findall("./catalog[@kind='current']/book"):
    print(
        book.get("id"),                  # атрибут
        book.get("lang", "unknown"),     # у b2 атрибута нет -> значение по умолчанию
        book.findtext("title"),          # str, если элемент нашёлся; None при промахе
        float(book.findtext("price")),   # схемы нет, поэтому тип приводим сами
    )

И три поведения, которые надо помнить: `find` ищет один, `findall` —
все, а промах — это `None`, а не исключение.

In [ ]:
print(root.find(".//book[@id='b2']").findtext("title"))   # .// — на любой глубине
print(root.find(".//book[@id='missing']"))                # промах: None, а не ошибка
print(len(root.findall(".//book")))                       # все книги обоих каталогов

#### ❓ **Вопрос**: Чем `find(".//book[@id='b2']")` отличается от `findall("./catalog[@kind='current']/book")` и почему вокруг `findtext("price")` стоит именно `float`, а не `int`?

<details>

<summary><strong>Ответ</strong></summary>

`find` возвращает **один** элемент (первый совпавший) или `None`, а `findall` — **список**, возможно пустой. Пути тоже разные: `.//` ищет на любой глубине по всему дереву, `./catalog[@kind='current']/book` — строго по указанному пути и только в каталоге с нужным атрибутом. Приведение типа обязательно, потому что без схемы `ElementTree` отдаёт текст строкой: `findtext("price")` вернёт `"1200"`, и `900 + "1200"` уронит программу, а сортировка по цене отсортирует лексикографически. А `float`, а не `int`, потому что цена бывает дробной: в задаче B5 лежит `1200.50`, и `int("1200.50")` выбросит `ValueError` — `int()` не умеет читать точку в строке.

</details>

XML не только читают — его и пишут: этого просят задачи M2 и H2. Дерево
собирают из объектов: `ET.Element` создаёт корень, `ET.SubElement` — ребёнка
внутри указанного родителя. Атрибуты передают именованными аргументами, а
текст кладут в `.text`.

In [ ]:
out_root = ET.Element("students")                        # корень будущего документа
for index, (name, score) in enumerate([("Анна", 86), ("Илья", 73)], start=1):
    student = ET.SubElement(out_root, "student", id=f"s{index}")   # атрибут id
    ET.SubElement(student, "name").text = name
    ET.SubElement(student, "score").text = str(score)    # .text принимает только строку

Записываем дерево в файл — и сразу смотрим, что получилось.

In [ ]:
xml_path = work_path("students.xml")
tree = ET.ElementTree(out_root)
tree.write(xml_path, encoding="utf-8", xml_declaration=True)

show_text(xml_path)   # всё в одну строку: ElementTree по умолчанию не форматирует

Читать такое глазами тяжело. С Python 3.9 есть `ET.indent` — он
расставляет переносы и отступы прямо в дереве, перед записью.

In [ ]:
ET.indent(tree)       # правит дерево на месте, поэтому вызывать до write
tree.write(xml_path, encoding="utf-8", xml_declaration=True)

show_text(xml_path)   # тот же документ, но теперь его можно читать и ревьюить

<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

Отсутствие переносов и отступов на корректность не влияет — пробелы
между тегами для парсера незначимы, поэтому `ET.indent` появился только в
Python 3.9. До этого за отступами лезли в `xml.dom.minidom` или расставляли их
руками; в старом коде вы это ещё встретите.

Отдельная ловушка: `.text` принимает только строку. Присвоите туда число —
получите `TypeError` уже на `write`, а не на присваивании, и искать причину
будете не там, где ошиблись.

</details>

И закрываем круг: читаем свой же файл обратно. `ET.parse` работает с
файлом (в отличие от `ET.fromstring`), а `getroot()` достаёт корень.

In [ ]:
again = ET.parse(xml_path).getroot()      # parse — из файла, fromstring — из строки
print(again.findtext("student/score"))    # "86" в кавычках: текст снова пришёл строкой
print(again[0].get("id"))                 # атрибут первого ребёнка

#### ❓ **Вопрос**: Мы записали `students.xml` и тут же прочитали его обратно. Почему `findtext("student/score")` вернул `"86"` в кавычках, а не число 86?

<details>

<summary><strong>Ответ</strong></summary>

Потому что в XML текст элемента — всегда строка: типов у формата нет. Мы и записывали строкой (`str(score)` — иначе `write` упал бы с `TypeError`), и читаем строкой. Тип задаёт схема (XSD), а `ElementTree` схему не применяет, поэтому приведение к числу — работа вашего кода: `int(again.findtext("student/score"))`. В JSON эта проблема не возникает: там `86` уже число, что мы видели в первом разделе.

</details>

## 4. HTTP и REST: как попросить данные у чужой программы

API — это договор между клиентом и сервером: какой адрес дёрнуть, каким
методом, что передать, какие статусы и структуру ответа ожидать. REST —
самый распространённый стиль такого договора: URL обозначает **ресурс**
(`/users/42`), а HTTP-метод — **действие** над ним.

Два слова, которые постоянно путают:

- **безопасный** (safe) метод только читает и ничего на сервере не меняет;
- **идемпотентный** метод можно повторить: итоговое состояние сервера будет
  тем же (журнал и текст ответа при этом могут отличаться).

Безопасные и идемпотентные HTTP-методы

| Метод | Назначение | Безопасен | Идемпотентен | Тело запроса |
|---|---|---:|---:|---|
| `GET` | получить ресурс | да | да | нет |
| `HEAD` | только заголовки, как у `GET` | да | да | нет |
| `POST` | создать, изменить или запустить действие | нет | нет | обычно да |
| `PUT` | создать или целиком заменить ресурс по URL | нет | да | обычно да |
| `PATCH` | изменить часть ресурса | нет | нет | обычно да |
| `DELETE` | удалить ресурс | нет | да | обычно нет |
| `OPTIONS` | узнать возможности взаимодействия | да | да | обычно нет |
| `QUERY` | читающий запрос с описанием в теле | да | да | ожидается |

Практическое правило курса: **`POST` меняет состояние сервера и сам собой не
повторяется**. Если автор конкретного API сделал читающий `POST` — это его
частная договорённость, и по методу её распознать нельзя.

`QUERY` — свежий метод: идемпотентная замена `POST`, когда запрос слишком
велик для URL, но данные при этом только читаются. Не для создания и
изменения. Сервер может его ещё не поддерживать — в задаче H1 учебный сервер
поддерживает.

**Query-параметры URI** — это другое, несмотря на похожее слово: часть адреса
после `?`, вида `/items?category=book&page=2`. В `requests` их передают
словарём `params`, и библиотека сама всё закодирует.

Тело запроса передают аргументом `json=`: он сам сериализует словарь и
проставит заголовок `Content-Type: application/json`. Отправим `POST` и
посмотрим на него глазами сервера — это ровно то, что просят в H1.

Идём за данными. `httpbin.org` — публичная «зеркалка»: она возвращает
JSON с тем, что получила от вас, и потому удобна для демонстраций.

На занятии внешняя сеть отваливается регулярно, поэтому у семинара есть
дублёр: учебный сервер из `assets/api_server.py` отвечает на те же адреса
(`/get`, `/status/404`, `/html`, `/anything`). Поднимите его заранее, и
следующая ячейка сама переключится на него, если интернета не окажется:

```bash
cd ~/seminar-11 && ./generate.sh
uv run uvicorn api_server:app --app-dir assets --port 8000 &
```

In [ ]:
import requests

BASE = "https://httpbin.org"                   # публичная «зеркалка»
try:
    requests.get(f"{BASE}/get", timeout=5)     # проверяем, есть ли она сейчас
except requests.RequestException:              # нет сети или httpbin лежит
    BASE = "http://127.0.0.1:8000"             # ...идём на учебный сервер семинара
print("работаем через", BASE)                  # всегда видно, куда уходят запросы

Первый запрос. Обратите внимание на два аргумента: `params` собирает
строку после `?` и сам кодирует значения, `timeout` не даёт скрипту зависнуть
навсегда.

In [ ]:
response = requests.get(
    f"{BASE}/get",
    params={"course": "python", "page": 2},   # requests сам соберёт ?course=python&page=2
    timeout=10,                               # без таймаута запрос может висеть вечно
)

Ответ получен — смотрим, что именно ушло на сервер и что он ответил.

In [ ]:
print(response.status_code)   # 200 — HTTP-уровень отработал
print(response.url)           # итоговый URL: видно, как params превратились в ?...
print(response.json()["args"])# httpbin вернул наши параметры — все значения строками

Теперь `POST`. Адрес `/anything` — то же зеркало, но принимающее тело
запроса: он вернёт нам метод, который увидел, и разобранный JSON.

In [ ]:
answer = requests.post(
    f"{BASE}/anything",
    json={"category": "books", "max_price": 1500},   # json= сам сериализует словарь
    timeout=10,
).json()
print(answer["method"])   # POST — сервер сообщает, каким методом к нему пришли
print(answer["json"])     # и отдаёт тело обратно уже разобранным

А теперь тот же запрос методом `QUERY`. Отдельной функции `requests.query`
нет — для нестандартного метода есть общий `requests.request`.

In [ ]:
response = requests.request(
    "QUERY",                                         # метод строкой — так можно любой
    f"{BASE}/anything",
    json={"category": "books", "max_price": 1500},
    timeout=10,
)
print(response.status_code)     # 405 Method Not Allowed: этот сервер QUERY не умеет

`405` — не ошибка вашего кода, а ответ «такой метод здесь не разрешён»:
`QUERY` слишком новый, и поддерживают его пока единицы. В задаче H1 учебный
сервер семинара его поддерживает — на адресе `/echo`, и там же видно, что тело
у `QUERY` и `POST` одинаковое.

<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

Всегда ставьте `timeout`. Умолчание в `requests` — ждать бесконечно:
если сервер принял соединение и замолчал, ваш скрипт повиснет навсегда, и в
логах это выглядит как «задача просто не закончилась». Классика — ночной
пайплайн, который вместо часа шёл до утра, потому что один из тысячи запросов
ушёл в никуда.

И заметьте, что `params` кодирует значения за вас: пробелы, кириллицу,
амперсанды. Руками собранная строка `url + "?q=" + text` ломается на первом же
пользовательском вводе со знаком `&` — а иногда и не ломается, а тихо
отправляет не то, что вы думали.

</details>

#### ❓ **Вопрос**: Соединение оборвалось до того, как пришёл ответ. Какие из `GET`, `POST` и `QUERY` программа может повторить сама, а какой — нет и почему?

<details>

<summary><strong>Ответ</strong></summary>

Сама повторять можно `GET` и `QUERY`: оба безопасны, то есть только читают, и потому идемпотентны — второй такой же запрос ничего не изменит. `POST` повторять нельзя: он не идемпотентен, и мы не знаем, дошёл ли первый запрос до сервера. Может оказаться, что заказ уже создан или задача уже запущена, и повтор сделает второй. Такой запрос повторяет человек, разобравшись, что произошло.

</details>

## 5. Четыре разных смысла слова «ошибка»

Фраза «запрос не сработал» ничего не говорит, пока не назван этаж, на котором
всё сломалось: чинить их надо по-разному, а проверять — строго по порядку.

Четыре этажа ошибок при работе с HTTP API

Первая цифра статуса задаёт класс ответа. **2xx — это успех, а не ошибка.**
3xx — «иди по другому адресу», 4xx — виноват запрос клиента, 5xx — сервер или
промежуточный прокси.

Про 3xx важно знать сразу: `requests` **сам молча ходит по редиректам**, и в
`response.status_code` вы увидите уже 200 конечной страницы. Куда вас в итоге
привели — видно в `response.url` (мы это печатали выше), а всю цепочку хранит
`response.history`. Отключается это `allow_redirects=False`.

| Статус | Название | Что обычно означает |
|---:|---|---|
| `200` | OK | выполнено, результат в теле |
| `201` | Created | ресурс создан |
| `204` | No Content | выполнено, тела нет |
| `301`/`302` | Moved Permanently / Found | ресурс переехал, идти по `Location` |
| `304` | Not Modified | с прошлого раза не менялось, берите из кэша |
| `400` | Bad Request | неверный синтаксис или параметры запроса |
| `401` | Unauthorized | токена нет или он неверный |
| `403` | Forbidden | сервер узнал клиента, но действие запрещено |
| `404` | Not Found | по этому URL ресурса нет |
| `422` | Unprocessable Content | формат понятен, данные не прошли валидацию |
| `429` | Too Many Requests | превышен лимит запросов |
| `500` | Internal Server Error | внутренняя ошибка сервера |
| `502` | Bad Gateway | промежуточный сервер получил плохой ответ |
| `503` | Service Unavailable | сервис временно недоступен |

Пройдём по этажам сверху вниз, каждый — вживую. Этаж 1: HTTP-ответа нет вовсе.
Стучимся на заведомо закрытый порт локальной машины.

In [ ]:
try:
    requests.get("http://127.0.0.1:9/items", timeout=3)   # порт 9 закрыт: ответа не будет
except requests.ConnectionError as error:
    print("этаж 1, транспорт:", type(error).__name__)     # статуса нет — его некому прислать

Этаж 2: ответ пришёл, но статус плохой. Важный момент — сам по себе
`requests` на 404 исключения **не бросает**: код 404 это нормальный ответ
сервера. Превращает статус в исключение только `raise_for_status()`.

In [ ]:
response = requests.get(f"{BASE}/status/404", timeout=10)
print("статус получен:", response.status_code)   # исключения ещё не было

try:
    response.raise_for_status()                  # вот эта строка и поднимает HTTPError
except requests.HTTPError as error:
    print("этаж 2, HTTP:", error.response.status_code)

Этаж 3: статус `200`, но в теле не JSON. Так выглядит подсунутая
провайдером страница-заглушка или HTML-ошибка от прокси.

In [ ]:
response = requests.get(f"{BASE}/html", timeout=10)
response.raise_for_status()                      # на уровне HTTP всё хорошо: 200

try:
    response.json()                              # а тело — HTML
except requests.exceptions.JSONDecodeError:
    print("этаж 3, формат:", response.headers["Content-Type"])

Этаж 4: и статус хороший, и JSON разобрался, а нужных полей нет. Здесь
уже никто не ругнётся — проверяет только ваш код.

In [ ]:
data = requests.get(f"{BASE}/get", timeout=10).json()

if not isinstance(data.get("items"), list):      # ждали список items, а его нет
    print("этаж 4, данные:", sorted(data))       # что реально пришло в ответе

Токен авторизации передают **заголовком**, а не query-параметром: URL
целиком попадает в логи сервера, в историю браузера и в журналы прокси.

```python
token = os.environ["API_TOKEN"]                  # ключ берём из окружения, не из кода
headers = {"Authorization": f"Bearer {token}"}
response = requests.get(url, headers=headers, timeout=10)
```

Итоговый порядок проверки, он же порядок этажей: получить ответ → проверить
статус → разобрать формат → проверить структуру. Пропустили этаж — получите
непонятное падение двумя экранами ниже, уже в коде обработки.

<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

Про 401 и 403 есть старая шутка, что названия перепутали местами:
`401 Unauthorized` на самом деле означает «не аутентифицирован» (мы не поняли,
кто ты), а `403 Forbidden` — «аутентифицирован, но не авторизован». Название
осталось из первых версий HTTP и уже не изменится.

Ещё один этаж, который мы не разбираем, но встретится: `429 Too Many
Requests`. У почти любого публичного API есть лимит, и правильная реакция на
429 — не «повторить сразу же в цикле», а подождать столько, сколько указано в
заголовке `Retry-After`.

</details>

#### ❓ **Вопрос**: Чем `401` отличается от `403`, и почему после статуса `200` всё равно нужно отдельно разбирать JSON и проверять структуру?

<details>

<summary><strong>Ответ</strong></summary>

`401` — сервер не принял данные авторизации: токена нет, он просрочен или неверен; имеет смысл проверить ключ и повторить. `403` — клиент опознан, но прав на это действие у него нет; повторять бессмысленно, нужен другой доступ. `200` подтверждает успех **только на уровне HTTP** — это этаж 2. Ниже ещё два: тело может оказаться HTML вместо JSON (этаж 3, мы это видели на `/html`) и JSON может разобраться, но без нужных полей (этаж 4) — там не бросит исключение никто, кроме вашего кода.

</details>

## 6. HTML: когда данные остались только на странице

Аккуратный XHTML можно прочитать XML-парсером. Но обычный веб-HTML живёт по
своим правилам: закрывающие теги местами необязательны, `<br>` вообще не
парный, и браузер обязан восстановить смысл даже из откровенно кривой
разметки. XML-парсер так не умеет — и это принципиальная разница, а не
недоработка.

Представим, что тот же каталог книг нам не отдали ни JSON-ответом, ни
XML-выгрузкой — он есть только страницей на сайте. Страница с типичными
вольностями: незакрытые `<p>`, одиночный `<br>`, плюс `<script>` и `<style>`,
которых на экране не видно, но в тексте документа они лежат.

In [ ]:
html = """<html><head>
  <style>.hidden { display: none }</style>
  <script>console.log("этого на экране не видно")</script>
</head><body>
  <h1>Каталог</h1>
  <p>Python — 1200<br>
  <p>Linux — 900
</body></html>"""

Сначала честно попробуем прочитать это XML-парсером из прошлого
раздела.

In [ ]:
try:
    ET.fromstring(html)              # XML требует закрыть каждый элемент
except ET.ParseError as error:
    print("XML-парсер:", error)      # споткнулся и назвал строку с колонкой

А теперь Beautiful Soup — надстройка над HTML-парсером. Она достраивает
дерево по правилам HTML: закрывает `<p>` там, где это подразумевается, и не
требует `</br>`.

Второй аргумент — какой именно парсер разбирает разметку, и выбор тут не
случайный. `"html.parser"` встроен в Python, ставить ничего не надо, и для
учебных и большинства рабочих задач его хватает. `"lxml"` (пакет `lxml`)
заметно быстрее и пригодится на больших выгрузках, а `"html5lib"` разбирает
совсем сломанную вёрстку в точности как браузер, но он самый медленный.
Начинайте со встроенного и меняйте, только когда упрётесь в скорость или в
страницу, которую он не осилил.

In [ ]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(html, "html.parser")   # второй аргумент — какой парсер разбирает разметку
print(soup.h1.get_text())                   # дерево построилось, несмотря на «кривизну»
print(len(soup.select("p")))                # 2: парсер сам закрыл первый абзац

Осталось получить видимый текст. Прямой `get_text()` вытащил бы и
JavaScript, и CSS — их надо сначала выкинуть из дерева.

In [ ]:
for node in soup.select("script, style"):   # CSS-селектор: оба тега за один проход
    node.decompose()                        # удалить узел вместе со всем содержимым

print(soup.get_text(" ", strip=True))       # " " — чем склеивать, strip — без лишних пробелов

Текст целиком нужен редко — обычно из страницы выбирают конкретные
куски. Для этого у Beautiful Soup есть CSS-селекторы, те же, что в браузере:
`select(".product")` вернёт **список** узлов, `select_one(".price")` — первый
или `None`. Соберём карточки товаров — ровно то, что просят задачи M5 и H3.

In [ ]:
page = """<div class="product"><span class="name">Python</span><a href="py.html">→</a></div>
<div class="product"><span class="name">Linux</span><a href="/books/linux.html">→</a></div>"""
shop = BeautifulSoup(page, "html.parser")

for card in shop.select(".product"):            # список всех карточек
    print(card.select_one(".name").get_text(),  # select_one — первый узел или None
          card.select_one("a").get("href"))     # get читает атрибут тега

селектор ищет по дереву узлов, а не по тексту

Селектор идёт по дереву, а не по строке: `select` отдаёт список узлов —
возможно пустой, `select_one` — узел или `None`. Поиск внутри узла не выходит
за его поддерево, поэтому `card.select_one(".name")` не заглянет в соседнюю
карточку.

Ссылки на странице почти всегда относительные (`py.html`,
`/books/linux.html`), а сохранить надо абсолютные — иначе по ним потом никуда
не перейти. Склеивает адреса `urljoin` из стандартной библиотеки: он сам
разбирается, отсчитывать ли от каталога страницы или от корня сайта. Это
понадобится в M4 и H5.

In [ ]:
from urllib.parse import urljoin

base = "https://example.test/catalog/page.html"   # адрес страницы, где нашли ссылку
for card in shop.select(".product"):
    href = card.select_one("a").get("href")
    print(href, "->", urljoin(base, href))        # относительный -> абсолютный

как склеиваются относительная ссылка и ссылка от корня

До сих пор мы разбирали страницу-строку, которую сами же и написали. В
жизни она приезжает по HTTP: `requests` отдаёт её текст в `response.text`, и
этот текст — ровно то, что ждёт `BeautifulSoup`. Вот и весь скрапинг: **забрать
и разобрать**.

In [ ]:
response = requests.get(f"{BASE}/html", timeout=10)   # тот же адрес, что на этаже 3
response.raise_for_status()                           # проверяем статус ДО разбора

live = BeautifulSoup(response.text, "html.parser")    # response.text — исходный HTML
print(live.h1.get_text())                             # заголовок настоящей страницы
print(len(response.text), "символов разметки")        # столько пришлось разобрать

В H5 страницы лежат не в сети, а на диске, и адрес у них не `https://`, а
`file://`. Собирать такие пути руками неудобно, поэтому берут `pathlib.Path`:
`resolve()` делает путь абсолютным, `parent / имя` строит соседний файл,
`read_text()` читает содержимое, а `as_uri()` превращает путь в `file://`-адрес
— тот самый, который затем скармливают `urljoin`.

In [ ]:
from pathlib import Path

page_path = Path(work_path("catalog.html")).resolve()   # абсолютный путь, без ".."
page_path.write_text(page, encoding="utf-8")            # кладём ту же страницу на диск

print(page_path.as_uri())                               # file:///...: адрес для urljoin
print(urljoin(page_path.as_uri(), "py.html"))           # сосед по каталогу

Дальше всё повторяется: `BeautifulSoup(page_path.read_text(...))` — и
разбор идёт точно так же, как у страницы из сети.

<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

Название Beautiful Soup — из «Алисы в Стране чудес», песенка
Черепахи Квази. «Tag soup» (суп из тегов) — жаргонное название кривой
HTML-разметки, вот библиотека и обещает сварить из этого супа что-то приличное.

Практическая сторона, о которой стоит сказать вслух: скрапинг — это последнее
средство. Сначала ищите API или готовую выгрузку, потому что вёрстка меняется
без предупреждения и ваш парсер сломается молча, отдав пустой список вместо
данных. И проверяйте `robots.txt` и условия использования сайта: массовые
запросы без пауз — это как минимум невежливо, а иногда и незаконно.

</details>

#### ❓ **Вопрос**: Почему `ET.fromstring(html)` на этой строке падает, а `BeautifulSoup(html, "html.parser")` строит дерево — и что именно убрали вызовы `decompose()`?

<details>

<summary><strong>Ответ</strong></summary>

XML требует, чтобы каждый элемент был явно закрыт, а вложенность была правильной; в нашей строке `<br>` не закрыт и `<p>` не закрыты, поэтому `ElementTree` останавливается с `ParseError`. HTML-парсер знает правила HTML: `<br>` одиночный по спецификации, а открытый `<p>` автоматически закрывается следующим `<p>` — отсюда и `len(soup.select("p")) == 2`. `decompose()` удалил из дерева узлы `<script>` и `<style>` целиком, вместе с JavaScript-кодом и CSS внутри, — поэтому в `get_text()` остался только тот текст, который виден на экране.

</details>

## 7. Тот же REST API, только к языковой модели

Библиотека `openai` — это обёртка над обычным HTTP-запросом, который мы уже
умеем делать руками. Ценность в том, что тот же самый протокол понимают не
только серверы OpenAI: «OpenAI-compatible» сервер — это любой сервер,
реализующий тот же формат запроса и ответа. Локальный vLLM, Ollama,
корпоративный шлюз, учебный сервер этого семинара — код клиента не меняется.

Один клиент, три разных OpenAI-совместимых сервера

Отсюда три значения, которые полностью определяют, куда пойдёт запрос:

- `OPENAI_BASE_URL` — базовый адрес API, который выдал сервер;
- `OPENAI_API_KEY` — токен авторизации;
- `OPENAI_MODEL` — имя модели, доступной именно на этом сервере.

**Ключи в код не пишем** — ни здесь, ни где-либо ещё: ноутбук уедет в git, а
оттуда ключ уже не вычистить. Задаём их переменными окружения перед запуском
Jupyter:

```bash
export OPENAI_BASE_URL='https://api.openai.com/v1/'
export OPENAI_API_KEY='ваш-ключ'
export OPENAI_MODEL='gpt-4o-mini'
```

Если ключа нет, эта часть демки всё равно запускается — поднимите учебный
сервер семинара, он отвечает в том же формате:

```bash
cd ~/seminar-11 && ./generate.sh
uv run uvicorn api_server:app --app-dir assets --port 8000 &
export OPENAI_BASE_URL='http://127.0.0.1:8000/v1'
export OPENAI_API_KEY='local'
export OPENAI_MODEL='seminar-chat'
```

`/v1` — часть адреса конкретного сервера, а не украшение: его нельзя ни
дописывать, ни убирать по своему вкусу.

In [ ]:
# Все три значения — из окружения. В коде ключа нет и быть не должно.
base_url = os.environ["OPENAI_BASE_URL"]
api_key = os.environ["OPENAI_API_KEY"]
model = os.environ["OPENAI_MODEL"]
print(base_url, model)          # ключ не печатаем: вывод ячейки сохранится в .ipynb

Запрос — это список сообщений с ролями. Роль `user` — то, что спросил
пользователь; ответ модели придёт с ролью `assistant`.

In [ ]:
from openai import OpenAI

client = OpenAI(base_url=base_url, api_key=api_key)
messages = [{"role": "user", "content": "Коротко поприветствуй студентов курса Python."}]

completion = client.chat.completions.create(model=model, messages=messages)

Ответ — объект, разобранный SDK из JSON. Полезного там больше, чем один
текст: видно, какая модель реально ответила и сколько токенов потрачено.

In [ ]:
print(completion.choices[0].message.content)   # сам текст ответа
print(completion.model)                        # какая модель ответила на самом деле
print(completion.usage.total_tokens)           # за это выставляют счёт

Под капотом SDK отправил обычный `POST` — ровно такой, какой мы умеем
собрать сами через `requests`. Полезно уметь: так отлаживают, когда SDK
скрывает причину ошибки.

```python
response = requests.post(
    f"{base_url.rstrip('/')}/chat/completions",
    headers={"Authorization": f"Bearer {api_key}"},
    json={"model": model, "messages": messages},
    timeout=30,
)
response.raise_for_status()
print(response.json()["choices"][0]["message"]["content"])
```

<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

Полезная привычка при работе с любым «совместимым» сервером: если
что-то не работает, первым делом проверьте `base_url`. Половина проблем — это
лишний или недостающий `/v1`, а сообщение об ошибке при этом приходит
абсолютно неинформативное (обычно 404 в HTML, то есть этаж 3 из прошлого
раздела).

Вторая половина — модель. Имя `gpt-4o-mini` на локальном vLLM не значит
ничего: сервер знает ровно те модели, которые в него загрузили. Список можно
спросить у самого сервера — `client.models.list()`, — но и этот endpoint
реализован не у всех.

</details>

#### ❓ **Вопрос**: Программа читает три переменные окружения. Что сломается, если `OPENAI_BASE_URL` указывает на локальный сервер, а `OPENAI_MODEL` остался от облака OpenAI — и почему ключ не печатают в вывод ячейки?

<details>

<summary><strong>Ответ</strong></summary>

Адрес и ключ подойдут, соединение установится, а вот на запрос настоящий сервер ответит ошибкой: имени такой модели он не знает. `base_url` и `model` — связанная пара, имя берут из документации или из `client.models.list()` того же сервера, куда идёт запрос. (Учебный сервер семинара нарочно нетребователен и ответит на любое имя — не принимайте это за общее правило.) Ключ не печатают, потому что вывод ячейки сохраняется прямо в файле `.ipynb`: напечатанный ключ уедет в git вместе с ноутбуком и станет достоянием всех, у кого есть доступ к репозиторию.

</details>

## Дополнительно

Ниже — то, что не понадобится на занятии, но пригодится в задачах повышенной
сложности и в работе.

### XML Schema (XSD)

JSON Schema описывает структуру JSON, XSD — структуру XML: какие элементы
допустимы, в каком порядке, какие атрибуты обязательны, какого типа значения.

```xml
<xs:element name="price" type="xs:decimal"/>
<xs:attribute name="id" type="xs:string" use="required"/>
```

`ElementTree` схемы не проверяет — для этого берут `lxml` или `xmlschema`.
Значения после чтения всё равно приводят к типам Python явно.

### Полный XPath через lxml

`ElementTree` покрывает простые пути. Сравнения, функции, выбор текста и
атрибутов — это уже полный XPath, он есть в `lxml` (нужен в задаче M1):

```bash
uv add lxml
```

```python
from lxml import etree

document = etree.parse("books.xml")
titles = document.xpath(
    "/library/catalog[@kind='current']/book[price > 1000]/title/text()"
)
book_ids = document.xpath("//book/@id")
```

### HTTP-заголовки

`Accept` — какой формат ответа мы хотим, `Content-Type` — в каком формате наше
тело, `Authorization` — авторизация, `User-Agent` — кто мы такие.

```python
headers = {
    "Authorization": f"Bearer {os.environ['API_TOKEN']}",
    "Accept": "application/json",
}
response = requests.get(url, headers=headers, timeout=10)
```

Токен не пишут в код, не суют в query-параметры и не печатают в вывод.

### XPath по HTML вместо CSS-селекторов

Ещё один селектор, который пригодится в H3: `select("tbody tr")` перебирает
строки таблицы, а внутри строки `row.select("td")` — ячейки.

Beautiful Soup не поддерживает XPath. Даже в `BeautifulSoup(html, "lxml")`
слово `lxml` выбирает только парсер, а дерево остаётся деревом Beautiful Soup.
Если нужен именно XPath по HTML — читайте страницу через `lxml.html`:

```python
from lxml import html as lxml_html

tree = lxml_html.fromstring(html)
titles = tree.xpath("//article[@class='product']//h2/text()")
```

Одну страницу не разбирают сначала одним, потом другим: под задачу выбирают
либо CSS-селекторы Beautiful Soup, либо XPath `lxml`.

### YAML

YAML вы встретите в конфигурациях, CI, Docker Compose и Kubernetes. `.yml` и
`.yaml` — одно и то же. Формат допускает комментарии и обычно короче JSON, но
зависит от правильных отступов.

```bash
uv add pyyaml
```

```python
import yaml

with open("config.yml", encoding="utf-8") as file:
    config = yaml.safe_load(file)
```

Для чужих данных — только `safe_load`: обычный `load` умеет создавать
произвольные объекты Python, то есть исполнять чужой код. Дальше структуру и
типы проверяют так же, как у JSON.

### Пагинация и безопасное обновление кэша

Где заканчиваются страницы, каждый API решает сам: пустой список, поле `next`,
общее число страниц — правило берут из документации. Дубли снимают словарём по
`id`.

Проверенный ответ сначала пишут во временный файл и только потом
`os.replace("cache.json.tmp", "cache.json")`: `os.replace` атомарен, поэтому
падение на середине записи не испортит старый кэш.

В H5 `pathlib.Path` нужен для локальных HTML-файлов: `resolve()` даёт
абсолютный путь, `parent / href` — соседний файл, `as_uri()` превращает путь в
`file://` URL, от которого уже работает `urljoin`.

### OpenAI API: модели, роли, полный ответ

Список доступных моделей — `GET /v1/models`:

```python
models = client.models.list()

for item in models.data:
    print(item.id)
```

Полученный `id` и кладут в `OPENAI_MODEL`. Не каждый совместимый сервер этот
endpoint реализует; если приходит `404`, имя берут из документации сервера.

Роли в `messages`: `developer` задаёт правило работы (в старых версиях API это
называлось `system`), `user` — запрос пользователя, `assistant` — предыдущие
ответы модели. Историю диалога клиент хранит сам и отправляет целиком каждый
раз — сервер между запросами ничего не помнит (это понадобится в H4):

```python
messages = [
    {"role": "developer", "content": "Отвечай одним предложением."},
    {"role": "user", "content": "Что такое JSON?"},
]
```

Полный ответ удобно посмотреть целиком — `completion.model_dump_json(indent=2)`.
Интересны `choices` с результатами, `model` с фактически использованной
моделью, `usage` с токенами и `finish_reason` с причиной остановки
(`stop` — модель закончила сама, `length` — упёрлась в лимит токенов).

- [Список моделей OpenAI](https://developers.openai.com/api/docs/models)
- [Chat Completions API](https://developers.openai.com/api/reference/resources/chat/subresources/completions/methods/create)
- [Официальный quickstart OpenAI](https://developers.openai.com/api/docs/quickstart)

### Дополнительные материалы

- [json — документация Python](https://docs.python.org/3/library/json.html)
- [xml.etree.ElementTree — XPath support](https://docs.python.org/3/library/xml.etree.elementtree.html#xpath-support)
- [requests: Quickstart](https://requests.readthedocs.io/en/latest/user/quickstart/)
- [Beautiful Soup: документация](https://www.crummy.com/software/BeautifulSoup/bs4/doc/)
- [MDN: HTTP-методы](https://developer.mozilla.org/ru/docs/Web/HTTP/Methods) и
  [коды состояния](https://developer.mozilla.org/ru/docs/Web/HTTP/Status)